# set up

In [1]:
#set up
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from google.colab import drive
drive.mount('/content/drive')
data = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/02DataForAnalysis/')
result = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/03Results/')
brief = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/04DataForR/')

Mounted at /content/drive


# Install lib and dependencies

## example 1

In [3]:
!pip install sentence-transformers

In [4]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [5]:
model = SentenceTransformer("all-mpnet-base-v2")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
seq1 = 'I was not unaware of the problem.'
seq2 = 'I had a very slight awareness of the problem.'
embedding1 = model.encode(seq1, convert_to_tensor=True)
embedding2 = model.encode(seq2, convert_to_tensor=True)
cosine_scores = util.pytorch_cos_sim(embedding1, embedding2)
print("Sentence 1:", seq1)
print("Sentence 2:", seq2)
print("Similarity score:", cosine_scores.item())


Sentence 1: I was not unaware of the problem.
Sentence 2: I had a very slight awareness of the problem.
Similarity score: 0.7190778255462646


In [7]:
embeddings = model.encode([
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",])
similarities = model.similarity(embeddings, embeddings) #计算所有句子之间的两两相似度
similarities  #结果是一个相似度矩阵，用来衡量每对句子之间的语义相似程度。input=3个句子，output=3*3矩阵

tensor([[1.0000, 0.6817, 0.0492],
        [0.6817, 1.0000, 0.0421],
        [0.0492, 0.0421, 1.0000]])

## example 2

In [8]:
post ='who is the highest-grossing actor? samuel l. jackson reflects on his career and achievements in the cover story of the october/november issue of aarp the magazine. read the full article here:'


In [ ]:
post = "Mrs. Grove called from the door that opened towards the garden. But no answer came. The sun had set half an hour before, and his parting rays were faintly tinging with gold and purple, few clouds that lay just alone the edge of the western sky. In the east, the full moon was rising in all her beauty, making pale the stars that were sparking in the firmament."

In [ ]:
post = "I was not unaware of the problem. I had a very slight awareness of the problem."

In [9]:
# Step 1: Split text into sentences using spaCy
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp(post)
sentences = [sent.text.strip() for sent in doc.sents]
sentences

['who is the highest-grossing actor?',
 'samuel l. jackson reflects on his career and achievements in the cover story of the october/november issue of aarp the magazine.',
 'read the full article here:']

In [10]:
# Step 2: Encode each sentence into embeddings
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("all-mpnet-base-v2")  # 用这个模型生成embedding
embeddings = [model.encode(sentence, convert_to_tensor=True) for sentence in sentences]

In [11]:
# Step 3: Compute cosine similarities for consecutive sentence pairs
cosine_similarities = []
for i in range(len(embeddings) - 1):
    sim = util.pytorch_cos_sim(embeddings[i], embeddings[i + 1]).item()
    cosine_similarities.append(sim)

In [12]:
# Step 4: Calculate the average similarity
average_similarity = sum(cosine_similarities) / len(cosine_similarities) if cosine_similarities else None


In [13]:
# Output results
print("Sentences:", sentences)
print("Cosine Similarities:", cosine_similarities)
print("Average Similarity:", average_similarity)

Sentences: ['who is the highest-grossing actor?', 'samuel l. jackson reflects on his career and achievements in the cover story of the october/november issue of aarp the magazine.', 'read the full article here:']
Cosine Similarities: [0.2724628448486328, 0.14177252352237701]
Average Similarity: 0.2071176841855049


句子嵌入特性：

句子嵌入模型（如 all-mpnet-base-v2）生成的向量通常被标准化（归一化为单位向量），这意味着向量的模长为 1。
因此，余弦相似度的值主要集中在 [0, 1] 之间。
常见范围：

[0, 1] 是大多数自然语言处理中相似度的实际范围：

1: 两个句子语义完全相同。

≈0.5: 两个句子语义有一定相关性。

0: 两个句子语义完全不相关。

# define a function

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer, util

def process_text_similarity(data, text_column='clean_text'):
    """
    Processes text data to compute sentence embeddings, pairwise cosine similarities,
    and average similarity for each row in a DataFrame.

    Parameters:
    - data (pd.DataFrame): The input DataFrame containing text data.
    - text_column (str): The column name containing the clean text.

    Returns:
    - pd.DataFrame: The input DataFrame with additional columns:
        - 'sent_list': Tokenized sentences.
        - 'sentence_embeddings': Embeddings for each sentence.
        - 'pair_similarity': Pairwise cosine similarities between consecutive sentences.
        - 'average_similarity': Average pairwise similarity for each row.
    """
    # Ensure data is a DataFrame
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data) # Convert data to DataFrame if it's not

    # Step 1: Split text into sentences
    def split_into_sentences(text):
        return [sent.text.strip() for sent in nlp(text).sents]

    data['sent_list'] = data[text_column].apply(split_into_sentences)

    # Step 2: Encode sentences into embeddings
    model = SentenceTransformer("all-mpnet-base-v2")
    data['sentence_embeddings'] = data['sent_list'].apply(
        lambda sentences: [model.encode(sentence, convert_to_tensor=True) for sentence in sentences]
    )

    # Step 3: Compute pairwise cosine similarities
    def compute_pairwise_similarity(embeddings):
        if len(embeddings) < 2:
            return []  # No pairs to compare if fewer than 2 sentences
        similarities = []
        for i in range(len(embeddings) - 1):
            sim = util.pytorch_cos_sim(embeddings[i], embeddings[i + 1]).item()
            similarities.append(sim)
        return similarities

    data['pair_similarity'] = data['sentence_embeddings'].apply(compute_pairwise_similarity)

    # Step 4: Calculate the average similarity
    data['average_similarity'] = data['pair_similarity'].apply(
        lambda sims: np.mean(sims) if sims else None
    )

    return data


In [15]:
import pandas as pd

data = {
    'clean_text': [
        "The weather is nice. The sun is shining.",
        "I love programming. It's so much fun.",
        "This is an example text."
    ]
}

df = pd.DataFrame(data)


In [16]:
processed_df = process_text_similarity(df, text_column='clean_text')
processed_df

,clean_text,sent_list,sentence_embeddings,pair_similarity,average_similarity
0,The weather is nice. The sun is shining.,"[The weather is nice., The sun is shining.]","[[tensor(-0.0624, device='cuda:0'), tensor(-0....",[0.7358516454696655],0.735852
1,I love programming. It's so much fun.,"[I love programming., It's so much fun.]","[[tensor(-0.0127, device='cuda:0'), tensor(0.0...",[0.28367170691490173],0.283672
2,This is an example text.,[This is an example text.],"[[tensor(0.0023, device='cuda:0'), tensor(-0.1...",[],NaN


# ad (clinical group)

In [31]:
df = pd.read_csv(result + 'AD_clinical_result.csv',engine='python')
df

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,...,12.0,110.0,59.0,0.536364,11.0,0.916667,9.0,0.750000,3.0,0.250000
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,...,20.0,189.0,87.0,0.460317,16.0,0.800000,20.0,1.000000,3.0,0.150000
2,2,dementia-005-2,He_Hinzen,MildAD,55,male,okay he's falling off a chair . she's running ...,0.222,0.723,0.055,...,7.0,23.0,18.0,0.782609,0.0,0.000000,0.0,0.000000,0.0,0.000000
3,3,dementia-007-3,He_Hinzen,ModerateAD,75,female,well the girl is telling the boy to get the co...,0.054,0.893,0.053,...,9.0,87.0,54.0,0.620690,4.0,0.444444,8.0,0.888889,1.0,0.111111
4,4,dementia-010-0,He_Hinzen,MildAD,66,male,oh boy . wowie the boy's going up on a cookiej...,0.006,0.949,0.044,...,20.0,220.0,119.0,0.540909,19.0,0.950000,12.0,0.600000,11.0,0.550000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,190,Baycrest11976,Baycrest,MCI,88;00.,male,alright. you're going. uh the reason I'm laugh...,0.042,0.869,0.089,...,118.0,1030.0,299.0,0.290291,74.0,0.627119,93.0,0.788136,30.0,0.254237
191,191,Baycrest12257,Baycrest,MCI,74;00.,female,right. oh gosh. I don't really know. I guess m...,0.024,0.898,0.078,...,189.0,1569.0,389.0,0.247929,93.0,0.492063,168.0,0.888889,43.0,0.227513
192,192,Baycrest12813,Baycrest,MCI,66;00.,male,thinking back tell you a story. I have an inte...,0.050,0.844,0.105,...,290.0,2076.0,513.0,0.247110,119.0,0.410345,186.0,0.641379,54.0,0.186207
193,193,Baycrest7352,Baycrest,MCI,71;00.,female,can you repeat the question? okay well as I si...,0.034,0.863,0.103,...,39.0,397.0,206.0,0.518892,23.0,0.589744,47.0,1.205128,24.0,0.615385


In [ ]:
import re
def clean_punctuation_spacing(text):
    text = re.sub(r'\s+([.,!?;])', r'\1', text) #去掉标点符号前的空格，例如 'chair .' 变成 'chair.'
    text = text.replace('?', '.').replace('!', '.') #将 ? 和 ! 替换为 .，确保句子分割统一
    return text

# 对 `clean_text` 列进行清理
df['clean_text'] = df['clean_text'].apply(clean_punctuation_spacing)
df

In [32]:
data = df[['idx', 'PAR', 'clean_text']]
data.head()

,idx,PAR,clean_text
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...
1,1,dementia-003-0,here's a cookie jar . and the lid is off the c...
2,2,dementia-005-2,okay he's falling off a chair . she's running ...
3,3,dementia-007-3,well the girl is telling the boy to get the co...
4,4,dementia-010-0,oh boy . wowie the boy's going up on a cookiej...


In [33]:
data.shape

(195, 3)

In [34]:
df1 = process_text_similarity(data, text_column='clean_text')
df1.head()

<ipython-input-19-7583f42451ce>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sent_list'] = data[text_column].apply(split_into_sentences)
<ipython-input-19-7583f42451ce>:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentence_embeddings'] = data['sent_list'].apply(
<ipython-input-19-7583f42451ce>:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http

,idx,PAR,clean_text,sent_list,sentence_embeddings,pair_similarity,average_similarity
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,"[mhm ., there's a young boy going in a cookie ...","[[tensor(-0.0533, device='cuda:0'), tensor(-0....","[0.06052546575665474, 0.2375401109457016, 0.31...",0.256047
1,1,dementia-003-0,here's a cookie jar . and the lid is off the c...,"[here's a cookie jar ., and the lid is off the...","[[tensor(0.0140, device='cuda:0'), tensor(0.05...","[0.6875123977661133, 0.18274298310279846, 0.04...",0.249428
2,2,dementia-005-2,okay he's falling off a chair . she's running ...,"[okay he's falling off a chair ., she's runnin...","[[tensor(-0.0136, device='cuda:0'), tensor(0.0...","[0.20828530192375183, 0.1702633500099182, 0.17...",0.174167
3,3,dementia-007-3,well the girl is telling the boy to get the co...,[well the girl is telling the boy to get the c...,"[[tensor(0.0149, device='cuda:0'), tensor(-0.0...","[0.44014355540275574, 0.5047906637191772, 0.59...",0.372560
4,4,dementia-010-0,oh boy . wowie the boy's going up on a cookiej...,"[oh boy ., wowie the boy's going up on a cooki...","[[tensor(0.0064, device='cuda:0'), tensor(0.00...","[0.12643316388130188, 0.21444618701934814, 0.1...",0.204574


In [35]:
df['average_similarity'] = df1['average_similarity']
df

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,...,110.0,59.0,0.536364,11.0,0.916667,9.0,0.750000,3.0,0.250000,0.256047
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,...,189.0,87.0,0.460317,16.0,0.800000,20.0,1.000000,3.0,0.150000,0.249428
2,2,dementia-005-2,He_Hinzen,MildAD,55,male,okay he's falling off a chair . she's running ...,0.222,0.723,0.055,...,23.0,18.0,0.782609,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.174167
3,3,dementia-007-3,He_Hinzen,ModerateAD,75,female,well the girl is telling the boy to get the co...,0.054,0.893,0.053,...,87.0,54.0,0.620690,4.0,0.444444,8.0,0.888889,1.0,0.111111,0.372560
4,4,dementia-010-0,He_Hinzen,MildAD,66,male,oh boy . wowie the boy's going up on a cookiej...,0.006,0.949,0.044,...,220.0,119.0,0.540909,19.0,0.950000,12.0,0.600000,11.0,0.550000,0.204574
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,190,Baycrest11976,Baycrest,MCI,88;00.,male,alright. you're going. uh the reason I'm laugh...,0.042,0.869,0.089,...,1030.0,299.0,0.290291,74.0,0.627119,93.0,0.788136,30.0,0.254237,0.281931
191,191,Baycrest12257,Baycrest,MCI,74;00.,female,right. oh gosh. I don't really know. I guess m...,0.024,0.898,0.078,...,1569.0,389.0,0.247929,93.0,0.492063,168.0,0.888889,43.0,0.227513,0.262327
192,192,Baycrest12813,Baycrest,MCI,66;00.,male,thinking back tell you a story. I have an inte...,0.050,0.844,0.105,...,2076.0,513.0,0.247110,119.0,0.410345,186.0,0.641379,54.0,0.186207,0.242106
193,193,Baycrest7352,Baycrest,MCI,71;00.,female,can you repeat the question? okay well as I si...,0.034,0.863,0.103,...,397.0,206.0,0.518892,23.0,0.589744,47.0,1.205128,24.0,0.615385,0.303729


In [41]:
# save the results to a new cvs. file
df.to_csv(brief + 'AD_clinical_result.csv', index=False)

# #health (clinical group)

In [42]:
health = pd.read_csv(result + 'Health_clinical_result.csv',engine='python')
health.head()

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.010,0.925,0.064,...,18.0,133.0,80.0,0.601504,8.0,0.444444,3.0,0.166667,5.0,0.277778
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.020,0.980,0.000,...,12.0,88.0,56.0,0.636364,4.0,0.333333,6.0,0.500000,2.0,0.166667
2,2,control-015-1,He_Hinzen,Control,67,female,okay . a little boy is stepping on a stool tha...,0.029,0.923,0.048,...,20.0,170.0,93.0,0.547059,8.0,0.400000,17.0,0.850000,8.0,0.400000
3,3,control-017-4,He_Hinzen,Control,71,female,are you ready ? well the sink is overflowing ....,0.090,0.827,0.083,...,25.0,240.0,117.0,0.487500,22.0,0.880000,11.0,0.440000,10.0,0.400000
4,4,control-021-1,He_Hinzen,Control,74,female,okay . the mother's washing the dishes and the...,0.007,0.963,0.030,...,12.0,170.0,96.0,0.564706,13.0,1.083333,12.0,1.000000,7.0,0.583333


In [43]:
data2 = health[['idx', 'PAR', 'clean_text']]
data2.head()

,idx,PAR,clean_text
0,0,control-002-0,the scene is in the kitchen . the mother is wi...
1,1,control-013-0,somebody's getting cookies out of the cookie j...
2,2,control-015-1,okay . a little boy is stepping on a stool tha...
3,3,control-017-4,are you ready ? well the sink is overflowing ....
4,4,control-021-1,okay . the mother's washing the dishes and the...


In [44]:
df2 = process_text_similarity(data2, text_column='clean_text')
df2.head()

<ipython-input-19-7583f42451ce>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sent_list'] = data[text_column].apply(split_into_sentences)
<ipython-input-19-7583f42451ce>:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentence_embeddings'] = data['sent_list'].apply(
<ipython-input-19-7583f42451ce>:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http

,idx,PAR,clean_text,sent_list,sentence_embeddings,pair_similarity,average_similarity
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,"[the scene is in the kitchen ., the mother is ...","[[tensor(-0.0176, device='cuda:0'), tensor(-0....","[0.3767416477203369, 0.09094075113534927, 0.27...",0.228958
1,1,control-013-0,somebody's getting cookies out of the cookie j...,[somebody's getting cookies out of the cookie ...,"[[tensor(0.0142, device='cuda:0'), tensor(0.03...","[0.4790789484977722, 0.21283914148807526, 0.29...",0.255113
2,2,control-015-1,okay . a little boy is stepping on a stool tha...,"[okay ., a little boy is stepping on a stool t...","[[tensor(0.0320, device='cuda:0'), tensor(-0.0...","[-0.04114643856883049, 0.4036627411842346, 0.6...",0.266617
3,3,control-017-4,are you ready ? well the sink is overflowing ....,"[are you ready ?, well the sink is overflowing...","[[tensor(0.0053, device='cuda:0'), tensor(0.01...","[0.1899808645248413, 0.23602142930030823, 0.40...",0.205096
4,4,control-021-1,okay . the mother's washing the dishes and the...,"[okay ., the mother's washing the dishes and t...","[[tensor(0.0320, device='cuda:0'), tensor(-0.0...","[0.03031051717698574, 0.22288298606872559, 0.3...",0.218544


In [45]:
health['average_similarity'] = df2['average_similarity']
health

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.010,0.925,0.064,...,133.0,80.0,0.601504,8.0,0.444444,3.0,0.166667,5.0,0.277778,0.228958
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.020,0.980,0.000,...,88.0,56.0,0.636364,4.0,0.333333,6.0,0.500000,2.0,0.166667,0.255113
2,2,control-015-1,He_Hinzen,Control,67,female,okay . a little boy is stepping on a stool tha...,0.029,0.923,0.048,...,170.0,93.0,0.547059,8.0,0.400000,17.0,0.850000,8.0,0.400000,0.266617
3,3,control-017-4,He_Hinzen,Control,71,female,are you ready ? well the sink is overflowing ....,0.090,0.827,0.083,...,240.0,117.0,0.487500,22.0,0.880000,11.0,0.440000,10.0,0.400000,0.205096
4,4,control-021-1,He_Hinzen,Control,74,female,okay . the mother's washing the dishes and the...,0.007,0.963,0.030,...,170.0,96.0,0.564706,13.0,1.083333,12.0,1.000000,7.0,0.583333,0.218544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,99,94-1,Delaware,Control,68;00.,female,yep,0.000,0.000,1.000,...,1.0,1.0,1.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,NaN
100,100,96-1,Delaware,Control,61;00.,female,yep,0.000,0.000,1.000,...,1.0,1.0,1.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,NaN
101,101,97-1,Delaware,Control,84;00.,female,yeah,0.000,0.000,1.000,...,1.0,1.0,1.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,NaN
102,102,98-1,Delaware,Control,63;00.,female,okay,0.000,0.000,1.000,...,1.0,1.0,1.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,NaN


In [46]:
# save the results to the cvs. file
health.to_csv(brief + 'Health_clinical_result.csv', index=False)

# health (social media group)

In [51]:
h = pd.read_csv(result + 'Health_social_result.csv',engine='python')
h

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,2.0,49.0,41.0,0.836735,5.0,2.50,6.0,3.000000,6.0,3.000000
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,4.0,98.0,72.0,0.734694,3.0,0.75,10.0,2.500000,12.0,3.000000
2,2,488.0,NaN,51.0,AARP,AARP,2024-10-02T10:05:42-07:00,2.0,6.971043e+14,en,...,3.0,58.0,50.0,0.862069,6.0,2.00,4.0,1.333333,6.0,2.000000
3,3,113.0,NaN,19.0,AARP,AARP,2024-10-01T06:34:55-07:00,0.0,6.971043e+14,en,...,3.0,32.0,27.0,0.843750,0.0,0.00,2.0,0.666667,5.0,1.666667
4,4,318.0,NaN,74.0,AARP,AARP,2024-09-24T13:01:49-07:00,1.0,6.971043e+14,en,...,4.0,69.0,55.0,0.797101,5.0,1.25,6.0,1.500000,9.0,2.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3068,3068,1.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-29T19:03:18-07:00,0.0,1.180081e+15,en,...,2.0,41.0,37.0,0.902439,3.0,1.50,4.0,2.000000,4.0,2.000000
3069,3069,0.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-28T05:46:32-07:00,0.0,1.180081e+15,en,...,1.0,40.0,32.0,0.800000,4.0,4.00,6.0,6.000000,4.0,4.000000
3070,3070,0.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-28T05:45:39-07:00,0.0,1.180081e+15,en,...,2.0,46.0,37.0,0.804348,2.0,1.00,6.0,3.000000,2.0,1.000000
3071,3071,2.0,NaN,0.0,asaging,American Society on Aging,2024-09-07T06:58:41-07:00,0.0,1.180081e+15,en,...,1.0,11.0,11.0,1.000000,0.0,0.00,0.0,0.000000,2.0,2.000000


In [54]:
data = h[['idx', 'id', 'clean_text']]
data.head()

,idx,id,clean_text
0,0,1.087023e+15,aarp volunteers gathered with user experience ...
1,1,1.527138e+15,taraji p. henson stars in the newly launched p...
2,2,8.547888e+14,howie mandel found his purpose doing standup c...
3,3,1.220167e+15,who is the highest-grossing actor? samuel l. j...
4,4,8.840548e+14,legendary drummer and singer sheila e. has man...


In [56]:
# Step 1: Split text into sentences
data['clean_text'] = data['clean_text'].fillna('').astype(str) #Ensure clean_text column contains only strings
data['sent_list'] = data['clean_text'].apply(lambda x: [sent.text.strip() for sent in nlp(x).sents])

<ipython-input-56-487273bc5f2a>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['clean_text'] = data['clean_text'].fillna('').astype(str) #Ensure clean_text column contains only strings
<ipython-input-56-487273bc5f2a>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sent_list'] = data['clean_text'].apply(lambda x: [sent.text.strip() for sent in nlp(x).sents])


In [57]:
# Step 2: Compute Sentence Embeddings
model = SentenceTransformer("all-mpnet-base-v2")

data['sentence_embeddings'] = data['sent_list'].apply(
    lambda sentences: [model.encode(sentence, convert_to_tensor=True) for sentence in sentences])


<ipython-input-57-160a1566c7e6>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentence_embeddings'] = data['sent_list'].apply(


In [58]:
# Step 3: Compute Cosine Similarities
#Using the sentence embeddings, compute the cosine similarities between consecutive sentences for each row.
def compute_pairwise_similarity(embeddings):
    if len(embeddings) < 2:
        return []  # No pairs to compare if fewer than 2 sentences
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = util.pytorch_cos_sim(embeddings[i], embeddings[i + 1]).item()
        similarities.append(sim)
    return similarities

data['pair_similarity'] = data['sentence_embeddings'].apply(compute_pairwise_similarity)


<ipython-input-58-22306e359e31>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['pair_similarity'] = data['sentence_embeddings'].apply(compute_pairwise_similarity)


In [60]:
# Step 4: Calculate the average similarity
data['average_similarity'] = data['pair_similarity'].apply(lambda sims: np.mean(sims) if sims else None)
data

<ipython-input-60-00dbaae64675>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['average_similarity'] = data['pair_similarity'].apply(lambda sims: np.mean(sims) if sims else None)


,idx,id,clean_text,sent_list,sentence_embeddings,pair_similarity,average_similarity
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,[aarp volunteers gathered with user experience...,"[[tensor(0.0499, device='cuda:0'), tensor(0.02...",[0.6087697744369507],0.608770
1,1,1.527138e+15,taraji p. henson stars in the newly launched p...,[taraji p. henson stars in the newly launched ...,"[[tensor(0.0304, device='cuda:0'), tensor(0.06...","[0.2943909168243408, 0.4347154498100281, 0.509...",0.412788
2,2,8.547888e+14,howie mandel found his purpose doing standup c...,[howie mandel found his purpose doing standup ...,"[[tensor(0.0169, device='cuda:0'), tensor(0.05...","[0.450132817029953, 0.38892728090286255]",0.419530
3,3,1.220167e+15,who is the highest-grossing actor? samuel l. j...,"[who is the highest-grossing actor?, samuel l....","[[tensor(-0.0238, device='cuda:0'), tensor(0.0...","[0.2724628448486328, 0.14177252352237701]",0.207118
4,4,8.840548e+14,legendary drummer and singer sheila e. has man...,[legendary drummer and singer sheila e. has ma...,"[[tensor(-0.0279, device='cuda:0'), tensor(0.0...","[0.37493130564689636, 0.39795833826065063, 0.3...",0.366804
...,...,...,...,...,...,...,...
3068,3068,5.726419e+14,with more than 80% of older adults living with...,[with more than 80% of older adults living wit...,"[[tensor(0.0149, device='cuda:0'), tensor(0.08...",[0.1633610725402832],0.163361
3069,3069,9.777879e+14,"neither the systems serving older adults, nor ...","[neither the systems serving older adults, nor...","[[tensor(-0.0218, device='cuda:0'), tensor(0.0...",[],NaN
3070,3070,9.463259e+14,"in the fall of 2023, more than 40 indigenous e...","[in the fall of 2023, more than 40 indigenous ...","[[tensor(0.0209, device='cuda:0'), tensor(0.09...",[0.4618251323699951],0.461825
3071,3071,1.773792e+15,webinar tomorrow | learn more about later-life...,[webinar tomorrow | learn more about later-lif...,"[[tensor(0.0229, device='cuda:0'), tensor(0.09...",[],NaN


In [62]:
h['average_similarity'] = data['average_similarity']
h

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,49.0,41.0,0.836735,5.0,2.50,6.0,3.000000,6.0,3.000000,0.608770
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,98.0,72.0,0.734694,3.0,0.75,10.0,2.500000,12.0,3.000000,0.412788
2,2,488.0,NaN,51.0,AARP,AARP,2024-10-02T10:05:42-07:00,2.0,6.971043e+14,en,...,58.0,50.0,0.862069,6.0,2.00,4.0,1.333333,6.0,2.000000,0.419530
3,3,113.0,NaN,19.0,AARP,AARP,2024-10-01T06:34:55-07:00,0.0,6.971043e+14,en,...,32.0,27.0,0.843750,0.0,0.00,2.0,0.666667,5.0,1.666667,0.207118
4,4,318.0,NaN,74.0,AARP,AARP,2024-09-24T13:01:49-07:00,1.0,6.971043e+14,en,...,69.0,55.0,0.797101,5.0,1.25,6.0,1.500000,9.0,2.250000,0.366804
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3068,3068,1.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-29T19:03:18-07:00,0.0,1.180081e+15,en,...,41.0,37.0,0.902439,3.0,1.50,4.0,2.000000,4.0,2.000000,0.163361
3069,3069,0.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-28T05:46:32-07:00,0.0,1.180081e+15,en,...,40.0,32.0,0.800000,4.0,4.00,6.0,6.000000,4.0,4.000000,NaN
3070,3070,0.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-28T05:45:39-07:00,0.0,1.180081e+15,en,...,46.0,37.0,0.804348,2.0,1.00,6.0,3.000000,2.0,1.000000,0.461825
3071,3071,2.0,NaN,0.0,asaging,American Society on Aging,2024-09-07T06:58:41-07:00,0.0,1.180081e+15,en,...,11.0,11.0,1.000000,0.0,0.00,0.0,0.000000,2.0,2.000000,NaN


In [64]:
# save the results to a new cvs. file
h.to_csv(brief + 'Health_social_result.csv', index=False)

# ad (social media grop)

In [63]:
ad = pd.read_csv(result + 'AD_social_result.csv',engine='python')
ad.head()

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,25.0,363.0,197.0,0.542700,28.0,1.120000,26.0,1.040000,28.0,1.120000
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,14.0,294.0,177.0,0.602041,25.0,1.785714,28.0,2.000000,24.0,1.714286
2,2,2024-09-13T01:00:06-07:00,5.042727e+14,False,en,NaN,NaN,NaN,NaN,albums,...,3.0,50.0,42.0,0.840000,9.0,3.000000,4.0,1.333333,1.0,0.333333
3,3,2024-09-12T01:00:01-07:00,1.286549e+15,False,en,NaN,NaN,NaN,NaN,photos,...,31.0,573.0,282.0,0.492147,52.0,1.677419,62.0,2.000000,39.0,1.258065
4,4,2024-09-09T05:00:01-07:00,1.231950e+15,False,en,NaN,NaN,NaN,NaN,photos,...,15.0,309.0,173.0,0.559871,36.0,2.400000,33.0,2.200000,16.0,1.066667


In [66]:
data = ad[['idx', 'id', 'clean_text']]
data.head()

,idx,id,clean_text
0,0,1.194189e+15,shocking new data has shown that people living...
1,1,8.765100e+14,‘i never imagined dementia impacting our lives...
2,2,5.042727e+14,we know there is a long way to go when it come...
3,3,1.286549e+15,‘it felt like we had just been given a label a...
4,4,1.231950e+15,"‘we had all heard about dementia, but we never..."


In [68]:
# Step 1: Split text into sentences
data['clean_text'] = data['clean_text'].fillna('').astype(str) #Ensure clean_text column contains only strings
data['sent_list'] = data['clean_text'].apply(lambda x: [sent.text.strip() for sent in nlp(x).sents])

<ipython-input-68-487273bc5f2a>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['clean_text'] = data['clean_text'].fillna('').astype(str) #Ensure clean_text column contains only strings
<ipython-input-68-487273bc5f2a>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sent_list'] = data['clean_text'].apply(lambda x: [sent.text.strip() for sent in nlp(x).sents])


In [69]:
# Step 2: Compute Sentence Embeddings
model = SentenceTransformer("all-mpnet-base-v2")

data['sentence_embeddings'] = data['sent_list'].apply(
    lambda sentences: [model.encode(sentence, convert_to_tensor=True) for sentence in sentences])


<ipython-input-69-160a1566c7e6>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentence_embeddings'] = data['sent_list'].apply(


In [70]:
# Step 3: Compute Cosine Similarities
#Using the sentence embeddings, compute the cosine similarities between consecutive sentences for each row.
from sentence_transformers import util

def compute_pairwise_similarity(embeddings):
    if len(embeddings) < 2:
        return []  # No pairs to compare if fewer than 2 sentences
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = util.pytorch_cos_sim(embeddings[i], embeddings[i + 1]).item()
        similarities.append(sim)
    return similarities

data['pair_similarity'] = data['sentence_embeddings'].apply(compute_pairwise_similarity)


<ipython-input-70-9d6ddf081c8f>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['pair_similarity'] = data['sentence_embeddings'].apply(compute_pairwise_similarity)


In [72]:
# Step 4: Calculate the average similarity
data['average_similarity'] = data['pair_similarity'].apply(lambda sims: np.mean(sims) if sims else None)

<ipython-input-72-a8cc7f338ee5>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['average_similarity'] = data['pair_similarity'].apply(lambda sims: np.mean(sims) if sims else None)


In [73]:
data

,idx,id,clean_text,sent_list,sentence_embeddings,pair_similarity,average_similarity
0,0,1.194189e+15,shocking new data has shown that people living...,[shocking new data has shown that people livin...,"[[tensor(-0.0383, device='cuda:0'), tensor(0.0...","[0.544421911239624, 0.521043062210083, 0.31080...",0.375187
1,1,8.765100e+14,‘i never imagined dementia impacting our lives...,[‘i never imagined dementia impacting our live...,"[[tensor(-0.0116, device='cuda:0'), tensor(0.0...","[0.395562082529068, 0.20464003086090088, 0.529...",0.368396
2,2,5.042727e+14,we know there is a long way to go when it come...,[we know there is a long way to go when it com...,"[[tensor(0.0079, device='cuda:0'), tensor(0.11...","[0.5872561931610107, 0.48636072874069214]",0.536808
3,3,1.286549e+15,‘it felt like we had just been given a label a...,[‘it felt like we had just been given a label ...,"[[tensor(0.1090, device='cuda:0'), tensor(0.03...","[0.20319779217243195, 0.3089432716369629, 0.50...",0.329114
4,4,1.231950e+15,"‘we had all heard about dementia, but we never...","[‘we had all heard about dementia, but we neve...","[[tensor(0.0123, device='cuda:0'), tensor(0.05...","[0.22291894257068634, 0.20658959448337555, 0.1...",0.371303
...,...,...,...,...,...,...,...
2675,2675,5.187244e+14,want to know what the major parties are saying...,[want to know what the major parties are sayin...,"[[tensor(-0.0174, device='cuda:0'), tensor(0.0...",[0.11304961144924164],0.113050
2676,2676,3.725374e+15,don't miss out on one of the best races in the...,[don't miss out on one of the best races in th...,"[[tensor(-0.1348, device='cuda:0'), tensor(0.0...","[0.44366711378097534, 0.1759909987449646]",0.309829
2677,2677,8.861507e+14,'talking point' our online forum is 7 years ol...,['talking point' our online forum is 7 years o...,"[[tensor(0.0322, device='cuda:0'), tensor(0.00...",[],NaN
2678,2678,1.517941e+15,thanks to everyone that took part in this week...,[thanks to everyone that took part in this wee...,"[[tensor(-0.0266, device='cuda:0'), tensor(0.0...",[0.5220106840133667],0.522011


In [74]:
ad['average_similarity'] = data['average_similarity']
ad

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,363.0,197.0,0.542700,28.0,1.120000,26.0,1.040000,28.0,1.120000,0.375187
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,294.0,177.0,0.602041,25.0,1.785714,28.0,2.000000,24.0,1.714286,0.368396
2,2,2024-09-13T01:00:06-07:00,5.042727e+14,False,en,NaN,NaN,NaN,NaN,albums,...,50.0,42.0,0.840000,9.0,3.000000,4.0,1.333333,1.0,0.333333,0.536808
3,3,2024-09-12T01:00:01-07:00,1.286549e+15,False,en,NaN,NaN,NaN,NaN,photos,...,573.0,282.0,0.492147,52.0,1.677419,62.0,2.000000,39.0,1.258065,0.329114
4,4,2024-09-09T05:00:01-07:00,1.231950e+15,False,en,NaN,NaN,NaN,NaN,photos,...,309.0,173.0,0.559871,36.0,2.400000,33.0,2.200000,16.0,1.066667,0.371303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2675,2675,2010-04-15T07:17:43-07:00,5.187244e+14,False,NaN,alzheimers.org.uk,Alzheimer's Society called on our political pa...,http://www.alzheimers.org.uk/site/scripts/docu...,What are the three main political parties’ ele...,links,...,24.0,21.0,0.875000,2.0,1.000000,4.0,2.000000,2.0,1.000000,0.113050
2676,2676,2010-04-01T07:54:09-07:00,3.725374e+15,False,NaN,alzheimers.org.uk,NaN,http://www.alzheimers.org.uk/site/scripts/docu...,www.alzheimers.org.uk,links,...,32.0,29.0,0.906250,1.0,0.333333,2.0,0.666667,3.0,1.000000,0.309829
2677,2677,2010-03-31T08:29:41-07:00,8.861507e+14,False,NaN,forum.alzheimers.org.uk,"Many Happy Returns, TP Support for people with...",http://forum.alzheimers.org.uk/showthread.php?...,"Many Happy Returns, TP - Talking Point",links,...,9.0,9.0,1.000000,0.0,0.000000,0.0,0.000000,1.0,1.000000,NaN
2678,2678,2009-09-21T04:18:01-07:00,1.517941e+15,False,NaN,NaN,NaN,NaN,NaN,photos,...,26.0,24.0,0.923077,1.0,0.500000,2.0,1.000000,2.0,1.000000,0.522011


In [75]:
# save the results to a new cvs. file
ad.to_csv(brief + 'AD_social_result.csv', index=False)